In [ ]:
#HERE WE LOAD OUR RNN ORACLE

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
import json

# 1. Redefine your network architecture
class AnyLengthRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(AnyLengthRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, nonlinearity='relu')
        self.linear = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed_emb = pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed_emb)
        prediction = self.linear(h_n[-1, :, :])
        return prediction.squeeze()

# 2. Load the vocabulary
model_path = 'rnn_OhioT1DM.pth'
vocab_path = 'vocabulary_OhioT1DM.json'

with open(vocab_path, 'r') as f:
    loaded_vocab = json.load(f)

vocab_size = len(loaded_vocab) + 1  # +1 for Padding (0)
print(f"Loaded vocabulary: {loaded_vocab}")

# 3. Instantiate the empty model (with the same parameters used during training)
loaded_model = AnyLengthRNN(vocab_size=vocab_size, embed_dim=8, hidden_dim=32)

# 4. Fill the empty model with the saved weights
loaded_model.load_state_dict(torch.load(model_path))

# 5. CRITICAL! Set the model to Evaluation mode
loaded_model.eval()
print("✅ Model successfully loaded and ready for inference (or WFA extraction).")

# Assigning to final variables for later use
model = loaded_model
vocab = loaded_vocab

In [ ]:
#HERE WE COMPARE THE TWO APPROACHES FOR SOLVING EQUIVALENCE QUERIES

import os
import glob
import pickle
import itertools
import random
import matplotlib.pyplot as plt
import numpy as np
from TrainedRNNOracle import TrainedRNNOracle
from QuantitativeObservationTable import (
    QuantitativeObservationTable, 
    QuantitativeObservationTableParameters
)

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


oracle = TrainedRNNOracle(rnn_model=model, vocab_dict=vocab)
alphabet = ['N', 'M', 'I', 'E']

# SET TO CURRENT DIRECTORY: Looks in the same folder as the .ipynb file
file_path = "." 
lengths = list(range(1, 10))

# --- 1. Load the specific WFAs for this test ---
print(f"Looking for WFAs in the current directory...")
search_pattern = os.path.join(file_path, "WFA_*.pkl")
pkl_files = sorted(glob.glob(search_pattern))

# Debugging check to ensure files are actually being found
if not pkl_files:
    print(f"\n[!] ERROR: No files matching 'WFA_*.pkl' were found in the current directory.")
else:
    print(f"Found {len(pkl_files)} WFA files. Loading...")

loaded_wfas = {}
for path in pkl_files:
    with open(path, 'rb') as f:
        wfa = pickle.load(f)
        num_states = len(wfa.q0)
        file_name = os.path.basename(path)
        loaded_wfas[file_name] = wfa
        print(f" -> Loaded {file_name}: {num_states} states.")

# --- 2. Calculate Baseline Means and Generate Test Sets ---
print("\nGenerating test sets and calculating Baseline...")
baseline_means = {}
test_words_by_length = {}

for length in lengths:
    # Generate all possible combinations
    all_possible = [''.join(p) for p in itertools.product(alphabet, repeat=length)]
    random.shuffle(all_possible) # Shuffle to ensure randomness
    
    # a) Determine which words are used to calculate the mean weight
    if length >= 6:
        # For L>=6 use 2000 random words for the mean
        words_for_mean = all_possible[:2000]
        # Leave the rest for validation without repeating words
        validation_pool = all_possible[2000:] 
    else:
        # For L<6 use all for the mean
        words_for_mean = all_possible
        # Since we use all for the mean, overlapping for validation is unavoidable
        validation_pool = all_possible 
    
    # Calculate the mean weight (our "Baseline prediction" for this length)
    weights = [oracle.calculate_weight(w) for w in words_for_mean]
    length_mean = sum(weights) / len(weights)
    baseline_means[length] = length_mean
    print(f"  Length {length}: Baseline Mean = {length_mean:.2f} (calculated from {len(words_for_mean)} words)")
    
    # b) Determine the validation/test set
    if length <= 4:
        test_words = all_possible
        print(f"    -> Test: {len(test_words)} words (Exhaustive)")
    else: 
        # For lengths >= 5
        # Select 500 from the available pool
        test_words = random.sample(validation_pool, 500)
        print(f"    -> Test: 500 words (Randomized)")
        
    test_words_by_length[length] = test_words

# --- 3. Error Calculation (MAE) for WFAs and Baseline ---
print("\nCalculating errors against the Oracle. This might take a few seconds...")
wfa_errors = {name: [] for name in loaded_wfas.keys()}
baseline_errors = []

for length in lengths:
    test_words = test_words_by_length[length]
    pred_baseline = baseline_means[length]
    
    accumulated_baseline_error = 0.0
    accumulated_wfa_errors = {name: 0.0 for name in loaded_wfas.keys()}
    
    for word in test_words:
        pred_oracle = oracle.calculate_weight(word)
        
        # 1. Add baseline error
        accumulated_baseline_error += abs(pred_baseline - pred_oracle)
        
        # 2. Add WFA errors
        for name, wfa in loaded_wfas.items():
            pred_wfa = wfa.classify_word(word)
            
            if pred_wfa == float('-inf') and pred_oracle == float('-inf'):
                error_abs = 0.0
            elif pred_wfa == float('-inf') or pred_oracle == float('-inf'):
                error_abs = 100.0 
            else:
                error_abs = abs(pred_wfa - pred_oracle)
                
            accumulated_wfa_errors[name] += error_abs
            
    # Calculate averages for the current length
    baseline_errors.append(accumulated_baseline_error / len(test_words))
    for name in loaded_wfas.keys():
        wfa_errors[name].append(accumulated_wfa_errors[name] / len(test_words))

print("Calculations finished!")

# --- 4. Plotting the Error Curves ---
plt.figure(figsize=(10, 6))

# Plot WFAs
for name, wfa in loaded_wfas.items():
    num_states = len(wfa.q0)
    error_list = wfa_errors[name]
    
    # Customizing the legend based on the new file names
    name_lower = name.lower()
    
    if 'exhaustive' in name_lower:
        method = "Exhaustive search"
    elif 'abstraction' in name_lower:
        method = "Abstraction-based search"
    else:
        method = "Extracted Model"
        
    label = f"{method} ({num_states} states)"
    plt.plot(lengths, error_list, marker='o', label=label)

# Plot Baseline
plt.plot(lengths, baseline_errors, marker='s', color='red', linestyle='--', linewidth=2, label='Naïve Baseline (Length Average)')

# English titles and formatting
plt.title('Error Curves: Extracted WFAs & Baseline vs. RNN Oracle', fontsize=14)
plt.xlabel('Word Length', fontsize=12)
plt.ylabel('Mean Absolute Error (MAE)', fontsize=12)

plt.xticks(lengths)
plt.ylim(0, 50) 

plt.legend(title="Prediction Method", loc='upper left')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()

# --- 5. Save and Show the Image ---
save_path = os.path.join(file_path, "wfa_and_baseline_comparisons.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"\nPlot successfully saved at:\n{save_path}")

plt.show()

In [ ]:
"""
This script validates the WFA extraction framework against the clinical optimization 
queries presented in the Master Thesis using formal graph theory (Dynamic Programming over DAGs).

"""

import os
import time
import json
import pickle
import numpy as np
from typing import Dict, List, Optional, Tuple
from collections import deque
from scipy.sparse import coo_matrix

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

# Assuming TropicalWFA is available in the Python Path
# import sys
# sys.path.append(r'C:\Tu\Path\Al\Codigo')
from TropicalWFA import TropicalWFA 


# =============================================================================
# SECTION 1: RNN ORACLE DEFINITION & LOADING
# =============================================================================

class AnyLengthRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(AnyLengthRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, nonlinearity='relu')
        self.linear = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed_emb = pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed_emb)
        prediction = self.linear(h_n[-1, :, :])
        return prediction.squeeze()

def load_rnn_oracle(model_path: str, vocab_path: str):
    """Loads the pre-trained RNN and its vocabulary for empirical validation."""
    if not os.path.exists(model_path) or not os.path.exists(vocab_path):
        print(f"[!] Warning: RNN files not found. RNN empirical validation will be disabled.")
        return None, None
        
    with open(vocab_path, 'r') as f:
        vocab = json.load(f)
        
    vocab_size = len(vocab) + 1  # +1 for Padding (0)
    model = AnyLengthRNN(vocab_size=vocab_size, embed_dim=8, hidden_dim=32)
    model.load_state_dict(torch.load(model_path, weights_only=True))
    model.eval()  # CRITICAL: Set to evaluation mode
    return model, vocab

def evaluate_word_with_rnn(word: str, model: nn.Module, vocab: dict) -> float:
    """Evaluates a formal word sequence using the continuous RNN oracle."""
    if not model or not word: return 0.0
    with torch.no_grad():
        indices = [vocab[char] for char in word]
        t_in = torch.tensor(indices, dtype=torch.long).unsqueeze(0)
        t_len = torch.tensor([len(word)], dtype=torch.long)
        pred = model(t_in, t_len).item()
    return pred


# =============================================================================
# SECTION 2: CORE CLASSES (Data Structures & Graph Modeling)
# =============================================================================

class StructuralOracle:
    def __init__(self, length: Optional[int] = None, exact_counts: Optional[Dict[str, int]] = None, positions: Optional[Dict[int, str]] = None):
        self.length = length
        self.exact_counts = exact_counts or {}
        self.positions = positions or {}

class DFA:
    def __init__(self, alphabet: List[str], n_states: int, q0: int, final_states: List[int], transitions: Dict[str, List[tuple]]) -> None:
        self.alphabet = alphabet
        self.n_states = n_states
        self.q0 = q0
        self.final_states = set(final_states)
        self.delta = {a: np.zeros((n_states, n_states), dtype=bool) for a in alphabet}
        for a, trans_list in transitions.items():
            if a in self.delta:
                for src, dst in trans_list:
                    self.delta[a][src, dst] = True

class SparseTropicalWFA:
    def __init__(self, alphabet: List[str], q0: np.ndarray, final: np.ndarray, delta_sparse: Dict[str, 'csr_matrix'], zero_val: float):
        self.alphabet = alphabet
        self.q0 = q0          
        self.final = final    
        self.delta = delta_sparse 
        self.ZERO = zero_val
        self.n_states = len(q0)


# =============================================================================
# SECTION 3: GRAPH COMPILATION & INTERSECTION
# =============================================================================

def build_dfa_from_oracle(oracle: StructuralOracle, alphabet: List[str]) -> DFA:
    """Compiles a StructuralOracle into a strictly acyclic DFA."""
    tracked_chars = sorted(list(oracle.exact_counts.keys()))
    max_pos = oracle.length if oracle.length is not None else (max(oracle.positions.keys()) + 1 if oracle.positions else -1)
    
    q0_tuple = (0, tuple([0] * len(tracked_chars)))
    states_list = [q0_tuple]
    state_to_id = {q0_tuple: 0}
    transitions = {a: [] for a in alphabet}
    queue = [q0_tuple]

    def get_next_state(state: tuple, char: str):
        pos, counts = state
        if oracle.length is not None and pos >= oracle.length: return "DEAD"
        if pos in oracle.positions and oracle.positions[pos] != char: return "DEAD"
        
        next_pos = pos + 1 if oracle.length is not None else (pos + 1 if pos < max_pos else pos)
        next_counts = list(counts)
        if char in tracked_chars:
            idx = tracked_chars.index(char)
            next_counts[idx] += 1
            if next_counts[idx] > oracle.exact_counts[char]: return "DEAD"
        return (next_pos, tuple(next_counts))

    while queue:
        curr_state = queue.pop(0)
        curr_id = state_to_id[curr_state]
        for a in alphabet:
            nxt_state = get_next_state(curr_state, a)
            if nxt_state == "DEAD": continue 
                
            if nxt_state not in state_to_id:
                nxt_id = len(states_list)
                states_list.append(nxt_state)
                state_to_id[nxt_state] = nxt_id
                queue.append(nxt_state)
            else:
                nxt_id = state_to_id[nxt_state]
            transitions[a].append((curr_id, nxt_id))
            
    final_states = []
    for state, s_id in state_to_id.items():
        pos, counts = state
        is_final = True
        if oracle.length is not None and pos != oracle.length: is_final = False
        for i, char in enumerate(tracked_chars):
            if counts[i] != oracle.exact_counts[char]:
                is_final = False; break
        if is_final: final_states.append(s_id)

    return DFA(alphabet=alphabet, n_states=len(states_list), q0=0, final_states=final_states, transitions=transitions)

def intersect_wfa_dfa_sparse(wfa, dfa) -> SparseTropicalWFA:
    """Computes the Cartesian product of a WFA and DFA (yielding a Sparse DAG)."""
    new_alphabet = list(set(wfa.alphabet) & set(dfa.alphabet))
    start_w = np.where(wfa.q0 != wfa.ZERO)[0]
    start_d = dfa.q0
    
    state_map = {}
    queue = []
    for w in start_w:
        pair = (w, start_d)
        state_map[pair] = len(state_map)
        queue.append(pair)
        
    rows = {a: [] for a in new_alphabet}
    cols = {a: [] for a in new_alphabet}
    data = {a: [] for a in new_alphabet}
    
    idx = 0
    while idx < len(queue):
        u_w, u_d = queue[idx]
        u_id = state_map[(u_w, u_d)]
        idx += 1
        
        for a in new_alphabet:
            valid_v_d = np.where(dfa.delta[a][u_d])[0]
            if len(valid_v_d) == 0: continue 
            valid_v_w = np.where(wfa.delta[a][u_w] != wfa.ZERO)[0]
            if len(valid_v_w) == 0: continue 
                
            for v_d in valid_v_d:
                for v_w in valid_v_w:
                    weight = wfa.delta[a][u_w, v_w]
                    v_pair = (v_w, v_d)
                    if v_pair not in state_map:
                        state_map[v_pair] = len(state_map)
                        queue.append(v_pair)
                    v_id = state_map[v_pair]
                    rows[a].append(u_id)
                    cols[a].append(v_id)
                    data[a].append(weight)

    N_new = len(state_map)
    delta_sparse = {}
    for a in new_alphabet:
        mat_coo = coo_matrix((data[a], (rows[a], cols[a])), shape=(N_new, N_new), dtype=np.float64)
        delta_sparse[a] = mat_coo.tocsr()
        
    new_q0 = np.full(N_new, wfa.ZERO)
    new_final = np.full(N_new, wfa.ZERO)
    for w in start_w:
        pair = (w, start_d)
        if pair in state_map:
            new_q0[state_map[pair]] = wfa.q0[w]
            
    for (w, d), new_id in state_map.items():
        if d in dfa.final_states and wfa.final[w] != wfa.ZERO:
            new_final[new_id] = wfa.final[w]

    return SparseTropicalWFA(new_alphabet, new_q0, new_final, delta_sparse, wfa.ZERO)


# =============================================================================
# SECTION 4: TOPOLOGICAL ALGORITHMS (Extrema DP & Interval DP)
# =============================================================================

def _get_topological_order(sparse_wfa: SparseTropicalWFA) -> List[int]:
    n = sparse_wfa.n_states
    in_degree = np.zeros(n, dtype=int)
    for a in sparse_wfa.alphabet:
        mat = sparse_wfa.delta[a]
        in_degree += np.bincount(mat.indices, minlength=n)

    queue = deque(np.where(in_degree == 0)[0])
    top_order = []
    while queue:
        u = queue.popleft()
        top_order.append(u)
        for a in sparse_wfa.alphabet:
            mat = sparse_wfa.delta[a]
            for i in range(mat.indptr[u], mat.indptr[u+1]):
                v = mat.indices[i]
                in_degree[v] -= 1
                if in_degree[v] == 0: queue.append(v)
    return top_order

def find_extreme_weight_word(wfa: SparseTropicalWFA, find_max: bool = True) -> Tuple[Optional[str], float]:
    """Finds the absolute minimum or maximum weight path in the DAG."""
    n = wfa.n_states
    top_order = _get_topological_order(wfa)
    
    init_val = wfa.ZERO if find_max else np.inf
    dist = np.full(n, init_val, dtype=np.float64)
    parent = np.full(n, -1, dtype=int)
    sym_in = np.full(n, "", dtype=object)

    for i in range(n):
        if wfa.q0[i] != wfa.ZERO: dist[i] = wfa.q0[i]

    for u in top_order:
        if dist[u] == init_val: continue
        for a in wfa.alphabet:
            mat = wfa.delta[a]
            for i in range(mat.indptr[u], mat.indptr[u+1]):
                v = mat.indices[i]
                weight = mat.data[i]
                new_cost = dist[u] + weight
                if (find_max and new_cost > dist[v]) or (not find_max and new_cost < dist[v]):
                    dist[v] = new_cost
                    parent[v] = u
                    sym_in[v] = a

    extreme_total_weight = init_val
    best_final_state = -1
    for i in range(n):
        if wfa.final[i] != wfa.ZERO and dist[i] != init_val:
            total_weight = dist[i] + wfa.final[i]
            if (find_max and total_weight > extreme_total_weight) or (not find_max and total_weight < extreme_total_weight):
                extreme_total_weight = total_weight
                best_final_state = i

    if best_final_state == -1: return None, init_val

    curr = best_final_state
    word_reversed = []
    while parent[curr] != -1:
        word_reversed.append(sym_in[curr])
        curr = parent[curr]
    return "".join(reversed(word_reversed)), extreme_total_weight

def round_wfa_weights(wfa: SparseTropicalWFA) -> SparseTropicalWFA:
    """Rounds all weights in the WFA to integers to collapse the state space for interval searches."""
    new_q0 = np.where(wfa.q0 != wfa.ZERO, np.round(wfa.q0), wfa.ZERO)
    new_final = np.where(wfa.final != wfa.ZERO, np.round(wfa.final), wfa.ZERO)
    new_delta = {}
    for a in wfa.alphabet:
        mat = wfa.delta[a].copy()
        mat.data = np.round(mat.data)
        new_delta[a] = mat
    return SparseTropicalWFA(wfa.alphabet, new_q0, new_final, new_delta, wfa.ZERO)

def find_word_in_weight_interval(wfa: SparseTropicalWFA, min_w: float, max_w: float) -> Tuple[Optional[str], float]:
    """
    Finds a word whose total weight falls strictly within [min_w, max_w].
    By using integer weights, colliding paths merge and prevent exponential explosion.
    """
    n = wfa.n_states
    top_order = _get_topological_order(wfa)
    
    # dp[u] = {weight_accumulated: (parent_node, symbol, parent_weight)}
    dp = [{} for _ in range(n)]
    
    for i in range(n):
        if wfa.q0[i] != wfa.ZERO:
            dp[i][int(wfa.q0[i])] = (-1, "", -1)
            
    for u in top_order:
        if not dp[u]: continue
        for sym in wfa.alphabet:
            mat = wfa.delta[sym]
            for i in range(mat.indptr[u], mat.indptr[u+1]):
                v = mat.indices[i]
                weight = mat.data[i]
                if weight == wfa.ZERO: continue
                
                int_weight = int(weight)
                for current_weight in dp[u].keys():
                    new_weight = current_weight + int_weight
                    # We only need one valid path to reach this semantic future
                    if new_weight not in dp[v]:
                        dp[v][new_weight] = (u, sym, current_weight)

    best_final_state = -1
    best_weight_before_final = -1
    best_total_weight = -1
    
    for i in range(n):
        if wfa.final[i] != wfa.ZERO and dp[i]:
            final_weight_val = int(wfa.final[i])
            for w_accumulated in dp[i].keys():
                total_weight = w_accumulated + final_weight_val
                if min_w <= total_weight <= max_w:
                    best_final_state = i
                    best_weight_before_final = w_accumulated
                    best_total_weight = total_weight
                    break 
        if best_final_state != -1: break 
            
    if best_final_state == -1: return None, wfa.ZERO
        
    curr_node = best_final_state
    curr_w = best_weight_before_final
    word_reversed = []
    
    while True:
        parent, sym, p_weight = dp[curr_node][curr_w]
        if parent == -1: break 
        word_reversed.append(sym)
        curr_node = parent
        curr_w = p_weight
        
    return "".join(reversed(word_reversed)), float(best_total_weight)


# =============================================================================
# SECTION 5: MAIN CLINICAL EVALUATION BLOCK
# =============================================================================

def run_clinical_queries():
    alphabet = ['E', 'I', 'M', 'N']
    print("\n" + "="*80)
    print(" OHIO T1DM DATASET - CLINICAL OPTIMIZATION QUERIES")
    print("="*80)

    # Load the empirical RNN Oracle
    model, vocab = load_rnn_oracle('rnn_OhioT1DM.pth', 'vocabulary_OhioT1DM.json')

    # ---------------------------------------------------------
    # PART 1: NON-DETERMINISTIC WFA (Maximum Extrema Horizon N=6)
    # ---------------------------------------------------------
    nwfa_filename = "NWFA_651states.pkl"
    if os.path.exists(nwfa_filename):
        print(f"\n[+] Loading NWFA Model for Long-term Maxima Search: {nwfa_filename}")
        with open(nwfa_filename, "rb") as f:
            nwfa = pickle.load(f)
            
        print("\n--- NWFA Q1: Unconstrained Extrema (Safety Checks) ---")
        print("Q: How can we get the maximum glucose variation possible in 4 hours? (N=6)")
        oracle_1 = StructuralOracle(length=6)
        dag_1 = intersect_wfa_dfa_sparse(nwfa, build_dfa_from_oracle(oracle_1, alphabet))
        
        opt_word, opt_weight = find_extreme_weight_word(dag_1, find_max=True)
        rnn_val = evaluate_word_with_rnn(opt_word, model, vocab)
        
        print(f" -> Optimal Sequence: {opt_word}")
        print(f" -> WFA Prediction: {opt_weight:+.2f} mg/dL | RNN Empirical: {rnn_val:+.2f} mg/dL")

        print("\n--- NWFA Q2: Constrained Optimization (Treatment Planning) ---")
        print("Q: Exercise now, exactly 1 meal and 1 insulin in 4h. Maximize increase? (N=6)")
        oracle_2 = StructuralOracle(length=6, exact_counts={'M': 1, 'I': 1}, positions={0: 'E'})
        dag_2 = intersect_wfa_dfa_sparse(nwfa, build_dfa_from_oracle(oracle_2, alphabet))
        
        opt_word, opt_weight = find_extreme_weight_word(dag_2, find_max=True)
        rnn_val = evaluate_word_with_rnn(opt_word, model, vocab)
        
        print(f" -> Optimal Sequence: {opt_word}")
        print(f" -> WFA Prediction: {opt_weight:+.2f} mg/dL | RNN Empirical: {rnn_val:+.2f} mg/dL")
        
    else:
        print(f"\n[!] NWFA file {nwfa_filename} not found.")


    # ---------------------------------------------------------
    # PART 2: DETERMINISTIC WFA (Extrema and Targeted Intervals N=4)
    # ---------------------------------------------------------
    dwfa_filename = "DWFA_323states.pkl"
    if os.path.exists(dwfa_filename):
        print(f"\n\n[+] Loading DWFA Model for Absolute Minimums & Intervals: {dwfa_filename}")
        with open(dwfa_filename, "rb") as f:
            dwfa = pickle.load(f)
            
        print("\n--- DWFA Q3: Unconstrained Extrema (Safety Checks) ---")
        print("Q: Maximum and Minimum glucose variation possible in 2h 40m? (N=4)")
        oracle_3 = StructuralOracle(length=4)
        dag_3 = intersect_wfa_dfa_sparse(dwfa, build_dfa_from_oracle(oracle_3, alphabet))
        
        max_w, max_weight = find_extreme_weight_word(dag_3, find_max=True)
        min_w, min_weight = find_extreme_weight_word(dag_3, find_max=False)
        rnn_max = evaluate_word_with_rnn(max_w, model, vocab)
        rnn_min = evaluate_word_with_rnn(min_w, model, vocab)
        
        print(f" -> MAXIMUM Sequence: {max_w} (WFA: {max_weight:+.2f} | RNN: {rnn_max:+.2f})")
        print(f" -> MINIMUM Sequence: {min_w} (WFA: {min_weight:+.2f} | RNN: {rnn_min:+.2f})")

        print("\n--- DWFA Q4: Constrained Extrema (Treatment Planning) ---")
        print("Q: Exercise now, 1 meal, 0 insulin in 2h 40m. Maximize and Minimize increase? (N=4)")
        oracle_4 = StructuralOracle(length=4, exact_counts={'M': 1, 'I': 0}, positions={0: 'E'})
        dag_4 = intersect_wfa_dfa_sparse(dwfa, build_dfa_from_oracle(oracle_4, alphabet))
        
        max_w, max_weight = find_extreme_weight_word(dag_4, find_max=True)
        min_w, min_weight = find_extreme_weight_word(dag_4, find_max=False)
        rnn_max = evaluate_word_with_rnn(max_w, model, vocab)
        rnn_min = evaluate_word_with_rnn(min_w, model, vocab)
        
        print(f" -> MAXIMUM Sequence: {max_w} (WFA: {max_weight:+.2f} | RNN: {rnn_max:+.2f})")
        print(f" -> MINIMUM Sequence: {min_w} (WFA: {min_weight:+.2f} | RNN: {rnn_min:+.2f})")


        # ---------------------------------------------------------
        # PART 3: TARGETED INTERVALS (Integer-Bounded Dynamic Programming)
        # ---------------------------------------------------------
        print("\n\n[+] Rounding DAG Weights for Discrete Interval Resolution...")
        # We round the base DWFA to optimize intersection later
        dwfa_rounded = round_wfa_weights(dwfa)

        print("\n--- DWFA Q5: Neutral Regulation (Targeted Intervals) ---")
        print("Q: I will eat at t=0. What else to do to keep glucose variation between [-10, 10] mg/dL? (N=4)")
        oracle_5 = StructuralOracle(length=4, positions={0: 'M'})
        dag_5 = intersect_wfa_dfa_sparse(dwfa_rounded, build_dfa_from_oracle(oracle_5, alphabet))
        
        opt_word, opt_weight = find_word_in_weight_interval(dag_5, min_w=-10, max_w=10)
        rnn_val = evaluate_word_with_rnn(opt_word, model, vocab)
        print(f" -> Interval Sequence: {opt_word}")
        print(f" -> Rounded WFA Weight: {opt_weight:+.2f} mg/dL | RNN Empirical: {rnn_val:+.2f} mg/dL")

        print("\n--- DWFA Q6: Decrease Without Insulin (Targeted Intervals) ---")
        print("Q: Eat at t=0. Decrease glucose between [-25, -20] mg/dL strictly without insulin? (N=4)")
        oracle_6 = StructuralOracle(length=4, exact_counts={'I': 0}, positions={0: 'M'})
        dag_6 = intersect_wfa_dfa_sparse(dwfa_rounded, build_dfa_from_oracle(oracle_6, alphabet))
        
        opt_word, opt_weight = find_word_in_weight_interval(dag_6, min_w=-25, max_w=-20)
        rnn_val = evaluate_word_with_rnn(opt_word, model, vocab)
        print(f" -> Interval Sequence: {opt_word}")
        print(f" -> Rounded WFA Weight: {opt_weight:+.2f} mg/dL | RNN Empirical: {rnn_val:+.2f} mg/dL")
        
    else:
        print(f"\n[!] DWFA file {dwfa_filename} not found.")

    print("\n" + "="*80)
    print(" ALL QUERIES EXECUTED SUCCESSFULLY")
    print("="*80 + "\n")

if __name__ == "__main__":
    # To run this script, ensure the WFA pickle files, PyTorch model, and vocab are in the same directory.
    run_clinical_queries()

In [ ]:
#THIS CODE CAN BE USED FOR LEARNING NWFAs FROM THE RNN USING THE HYBRID-EXHAUSTIVE SEARCH METHOD FOR EQUIVALENCE QUERIES

import sys
import os

# Adjust this path to your local directory if necessary
project_path = r'C:\Users\rafam\Documents\GITI7\TFM\Codigo\TropicalSemirring_Functions'
if project_path not in sys.path:
    sys.path.append(project_path)

from QuantitativeObservationTable import (
    QuantitativeObservationTable, 
    QuantitativeObservationTableParameters
)

# Import the newly unified Oracle class
from TrainedRNNOracle import TrainedRNNOracle

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
vocab = {'N': 1, 'M': 2, 'I': 3, 'E': 4}

# 'model' is the RNN variable trained in the previous step
# We instantiate the unified Oracle with the specific tolerance from the dirty code
oracle = TrainedRNNOracle(rnn_model=model, vocab_dict=vocab, tol=40.0, rounding_step=1)

params = QuantitativeObservationTableParameters(
    tol_dist_init=20.0, 
    prefix_limit=50, 
    suffix_limit=25, 
    patience=5000, 
    rec_method="direct", 
    r=40, 
)

table = QuantitativeObservationTable(
    alphabet=oracle.alphabet,
    membership_query=oracle.calculate_weight, # Clean pointer to the RNN's MQ
    parameters=params
)

seen_counterexamples = set()
iteration = 1

print("Starting active learning loop with Neural Network Oracle...")

# ==========================================
# 2. MAIN L* LOOP
# ==========================================
while True:
    print(f"\n--- Iteration {iteration} ---")
    table.make_table_closed_and_consistent()
            
    # Reconstruct Hypothesis WFA
    wfa = table.reconstruct_WFA()
    
    # Equivalence Query using the unified method dispatcher
    ce = oracle.equivalence_query(
        wfa, 
        method="hybrid", 
        exhaustive_len=6, 
        random_max_len=10, 
        num_random=0
    )

    # Evaluate Equivalence Query Results
    if ce is None:
        print(f"\n[SUCCESS] No counterexamples found.")
        print(f"Extraction complete in {iteration} iterations. Total states: {len(table.S)}")
        break

    if ce in seen_counterexamples:
        print(f"\n[STOP] Repeated counterexample '{ce}' detected. Check tolerances or network precision.")
        break

    print(f"[+] Processing Counterexample: '{ce}' | S size: {len(table.S)}")

    # Process Counterexample using Binary Search as defined in the original code
    table.process_counterexample(ce, hypothesis_wfa=wfa, method="binary_search")
    seen_counterexamples.add(ce) 
    
    iteration += 1

In [ ]:
#THIS CODE CAN BE USED FOR LEARNING A NWFA FROM THE RNN USING THE ABSTRACTION BASED SEARCH METHOD

import sys
import os

# Adjust this path to your local directory if necessary
project_path = r'C:\Users\rafam\Documents\GITI7\TFM\Codigo\TropicalSemirring_Functions'
if project_path not in sys.path:
    sys.path.append(project_path)

from QuantitativeObservationTable import (
    QuantitativeObservationTable, 
    QuantitativeObservationTableParameters
)

# Import the newly unified Oracle class
from TrainedRNNOracle import TrainedRNNOracle

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
vocab = {'N': 1, 'M': 2, 'I': 3, 'E': 4}

# 'model' is the RNN variable trained in the previous step.
# We instantiate the unified Oracle with the specific tolerance from the dirty code.
# Note: max_eq_length is no longer passed here, it goes directly to the equivalence query.
oracle = TrainedRNNOracle(rnn_model=model, vocab_dict=vocab, tol=39.0, rounding_step=1)

params = QuantitativeObservationTableParameters(
    tol_dist_init=20.0, 
    prefix_limit=50, 
    suffix_limit=25, 
    patience=5000, 
    rec_method="direct", 
    r=40
)

table = QuantitativeObservationTable(
    alphabet=oracle.alphabet,
    membership_query=oracle.calculate_weight, # Clean pointer to the RNN's MQ
    parameters=params
)

seen_counterexamples = set()
iteration = 1

print("Starting active learning loop with Neural Network Oracle...")

# ==========================================
# 2. MAIN L* LOOP
# ==========================================
while True:
    print(f"\n--- Iteration {iteration} ---")
    table.make_table_closed_and_consistent()
            
    # Reconstruct Hypothesis WFA
    wfa = table.reconstruct_WFA()
    
    # Equivalence Query using the unified method dispatcher for Max-Plus Abstraction
    # We pass the max_eq_length and M_threshold directly here
    ce = oracle.equivalence_query(
        wfa, 
        method="abstraction", 
        max_eq_length=6, 
        M_threshold=10
    )

    # Evaluate Equivalence Query Results
    if ce is None:
        print(f"\n[SUCCESS] No counterexamples found.")
        print(f"Extraction complete in {iteration} iterations. Total states: {len(table.S)}")
        break

    if ce in seen_counterexamples:
        print(f"\n[STOP] Repeated counterexample '{ce}' detected. Check tolerances or network precision.")
        break

    print(f"[+] Processing Counterexample: '{ce}' | S size: {len(table.S)}")

    # Process Counterexample using Binary Search
    table.process_counterexample(ce, hypothesis_wfa=wfa, method="binary_search")
    seen_counterexamples.add(ce)  
    
    iteration += 1

In [ ]:
# THIS CODE CAN BE USED FOR LEARNING NWFAs FROM THE RNN USING THE HYBRID-EXHAUSTIVE SEARCH METHOD FOR EQUIVALENCE QUERIES 
# NOW WE INCLUDE THE MECHANISM TO FOCUS THE LEARNING IN SHORT WORDS

import sys
import os

# Adjust this path to your local directory if necessary
project_path = r'C:\Users\rafam\Documents\GITI7\TFM\Codigo\TropicalSemirring_Functions'
if project_path not in sys.path:
    sys.path.append(project_path)

from QuantitativeObservationTable import (
    QuantitativeObservationTable, 
    QuantitativeObservationTableParameters
)

from TrainedRNNOracle import TrainedRNNOracle

# ==========================================
# 1. TRUNCATION WRAPPER FOR THE ORACLE
# ==========================================
class TruncatedTrainedRNNOracle(TrainedRNNOracle):
    """
    Implementation of the pragmatic engineering trade-off:
    Prioritizes maximum precision over short temporal horizons to mitigate state explosion.
    Queries with a length greater than 'k_threshold' are truncated to their suffix of length 'k_threshold'.
    """
    def __init__(self, rnn_model, vocab_dict, tol, rounding_step, k_threshold):
        super().__init__(rnn_model=rnn_model, vocab_dict=vocab_dict, tol=tol, rounding_step=rounding_step)
        self.k = k_threshold
        
    def calculate_weight(self, word):
        """
        Overrides the Membership Query to evaluate only the suffix of length k.
        """
        if len(word) > self.k:
            word = word[-self.k:]
        return super().calculate_weight(word)
        
    def equivalence_query(self, wfa, method="hybrid", exhaustive_len=6, random_max_len=10, num_random=0):
        """
        Calls the base Equivalence Query. 
        Note: This assumes the base method internally uses `self.calculate_weight()`
        to evaluate the RNN targets, which will naturally inherit the truncation logic.
        """
        return super().equivalence_query(
            wfa, 
            method=method, 
            exhaustive_len=exhaustive_len, 
            random_max_len=random_max_len, 
            num_random=num_random
        )


# ==========================================
# 2. SETUP & CONFIGURATION
# ==========================================
vocab = {'N': 1, 'M': 2, 'I': 3, 'E': 4}

# Set the temporal horizon threshold
K_THRESHOLD = 6 

# 'model' is the RNN variable trained in the previous step
# Instantiate the extended Truncated Oracle
oracle = TruncatedTrainedRNNOracle(
    rnn_model=model, 
    vocab_dict=vocab, 
    tol=10.0, 
    rounding_step=1, 
    k_threshold=K_THRESHOLD
)

params = QuantitativeObservationTableParameters(
    tol_dist_init=10.0, 
    prefix_limit=50, 
    suffix_limit=25, 
    patience=5000, 
    rec_method="direct", 
    r=40, 
)

table = QuantitativeObservationTable(
    alphabet=oracle.alphabet,
    membership_query=oracle.calculate_weight, # Now safely points to the truncated logic
    parameters=params
)

seen_counterexamples = set()
iteration = 1

print(f"Starting active learning loop with Truncated Neural Network Oracle (k={K_THRESHOLD})...")

# ==========================================
# 3. MAIN L* LOOP
# ==========================================
while True:
    print(f"\n--- Iteration {iteration} ---")
    table.make_table_closed_and_consistent()
            
    # Reconstruct Hypothesis WFA
    wfa = table.reconstruct_WFA()
    
    # Equivalence Query using the unified method dispatcher
    ce = oracle.equivalence_query(
        wfa, 
        method="hybrid", 
        exhaustive_len=K_THRESHOLD, # Kept strictly aligned with the truncation horizon
        random_max_len=10, 
        num_random=0
    )

    # Evaluate Equivalence Query Results
    if ce is None:
        print(f"\n[SUCCESS] No counterexamples found.")
        print(f"Extraction complete in {iteration} iterations. Total states: {len(table.S)}")
        break

    if ce in seen_counterexamples:
        print(f"\n[STOP] Repeated counterexample '{ce}' detected. Check tolerances or network precision.")
        break

    print(f"[+] Processing Counterexample: '{ce}' | S size: {len(table.S)}")

    # Process Counterexample using Binary Search as defined in the original code
    table.process_counterexample(ce, hypothesis_wfa=wfa, method="binary_search")
    seen_counterexamples.add(ce) 
    
    iteration += 1

In [ ]:
# THIS CODE CAN BE USED FOR LEARNING DETERMINISTIC WFAs (DWFAs) FROM THE RNN 
# USING THE EXHAUSTIVE SEARCH METHOD AND TRUNCATED HORIZONS

import torch
import sys
import os


from QuantitativeObservationTable import (
    QuantitativeObservationTable, 
    QuantitativeObservationTableParameters
)

from TrainedRNNOracle import TrainedRNNOracle

# ==========================================
# 1. TRUNCATION WRAPPER FOR THE ORACLE
# ==========================================
class TruncatedTrainedRNNOracle(TrainedRNNOracle):
    """
    Implementation of the pragmatic engineering trade-off:
    Prioritizes maximum precision over short temporal horizons to mitigate state explosion.
    Queries with a length greater than 'k_threshold' are truncated to their suffix of length 'k_threshold'.
    """
    def __init__(self, rnn_model, vocab_dict, tol, rounding_step, k_threshold):
        super().__init__(rnn_model=rnn_model, vocab_dict=vocab_dict, tol=tol, rounding_step=rounding_step)
        self.k = k_threshold
        
    def calculate_weight(self, word):
        """
        Overrides the Membership Query to evaluate only the suffix of length k.
        """
        if len(word) > self.k:
            word = word[-self.k:]
        return super().calculate_weight(word)
        
    def equivalence_query(self, wfa, method="hybrid", exhaustive_len=4, random_max_len=10, num_random=0):
        """
        Calls the base Equivalence Query with the truncated logic.
        """
        return super().equivalence_query(
            wfa, 
            method=method, 
            exhaustive_len=exhaustive_len, 
            random_max_len=random_max_len, 
            num_random=num_random
        )

# ==========================================
# 2. SETUP & CONFIGURATION
# ==========================================
# Vocabulary for the OhioT1DM real-world application
vocab = {'N': 1, 'M': 2, 'I': 3, 'E': 4}

# For Deterministic WFAs, the paper states the horizon was reduced to N=4 (2h 40m)
K_THRESHOLD = 4 

# Instantiate the extended Truncated Oracle
oracle = TruncatedTrainedRNNOracle(
    rnn_model=model, 
    vocab_dict=vocab, 
    tol=5.0, 
    rounding_step=1,
    k_threshold=K_THRESHOLD
)

params = QuantitativeObservationTableParameters(
    tol_dist_init=10.0,  
    patience=5000
)

table = QuantitativeObservationTable(
    alphabet=oracle.alphabet,
    membership_query=oracle.calculate_weight, 
    parameters=params
)

seen_counterexamples = set()
iteration = 1

print(f"Starting active learning loop for DWFA with Truncated Oracle (k={K_THRESHOLD})...")

# ==========================================
# 3. MAIN L* LOOP (DETERMINISTIC)
# ==========================================
while True:
    print(f"\n--- Iteration {iteration} ---")
    
    # Specific method for Deterministic Table closure
    table.make_table_closed_and_consistent_det()
            
    # Reconstruct Deterministic Hypothesis WFA
    wfa = table.reconstruct_deterministic_wfa_opt()
    
    # Optional check (Useful to guarantee determinization properties)
    try:
        wfa.check_twins_property()
    except AttributeError:
        pass # Ignore if the method is not explicitly defined in the WFA class yet
    
    # Equivalence Query (Exhaustive search only, as justified in Section 1.3)
    ce = oracle.equivalence_query(
        wfa, 
        method="hybrid", 
        exhaustive_len=K_THRESHOLD, # Strict alignment with N=4
        random_max_len=15, 
        num_random=0 # Set to 0 because abstraction/random search was discarded for this application
    )

    # Evaluate Equivalence Query Results
    if ce is None:
        print(f"\n[SUCCESS] No counterexamples found.")
        print(f"Extraction complete in {iteration} iterations. Total states: {len(table.S)}")
        break

    if ce in seen_counterexamples:
        print(f"\n[STOP] Repeated counterexample '{ce}' detected. Check tolerances or network precision.")
        break

    print(f"[+] Processing Counterexample: '{ce}' | S size: {len(table.S)}")

    # Process Counterexample using 'all_suffixes' (Standard for DWFA extraction)
    table.process_counterexample(ce, hypothesis_wfa=wfa, method="all_suffixes")
    seen_counterexamples.add(ce)
            
    iteration += 1